In [0]:
-- ============================================
-- BRONZE 层: 增量数据加载 (使用 MERGE)
-- ============================================

-- 1. 创建临时视图连接 MySQL (使用 Databricks 的 MySQL 连接器)
CREATE OR REPLACE TEMPORARY VIEW mysql_order_detail_latest AS
SELECT 
    id          ,
    order_id        ,
    sku_id,
    sku_name,
    img_url,
    order_price,
    sku_num,
    create_time,
    split_total_amount ,
    split_activity_amount,
    split_coupon_amount,
    operate_time,
    CURRENT_TIMESTAMP() as _bronze_load_ts,
    'mysql_production' as _source_system,
    'order_detail' as _source_table
FROM awsmysql_catalog.gmall.order_detail
WHERE create_time > (
    SELECT COALESCE(MAX(create_time), '1900-01-01')
    FROM aws3.bronze.order_detail
);
---所以要定期-- 清理超过保留期限的文件（默认清理7天前的）我的表设置一天
--VACUUM your_table_name;实际数据在云上，但对DB，已经没了


-- 2. MERGE 到 Bronze 层
MERGE INTO aws3.bronze.order_detail AS target
USING mysql_order_detail_latest AS source
ON target.id = source.id 
   AND target._source_system = source._source_system
   AND target.create_time = source.create_time
WHEN NOT MATCHED THEN
INSERT (
    _bronze_load_ts,
    _bronze_load_id,
    _source_system,
    _source_table,
    id          ,
    order_id        ,
    sku_id,
    sku_name,
    img_url,
    order_price,
    sku_num,
    create_time,
    split_total_amount ,
    split_activity_amount,
    split_coupon_amount,
    operate_time
)
VALUES (
    source._bronze_load_ts,
    UUID() ,--as _bronze_load_id,
    source._source_system,
    source._source_table,
    source.id          ,
    source.order_id        ,
    source.sku_id,
    source.sku_name,
    source.img_url,
    source.order_price,
    source.sku_num,
    source.create_time,
    source.split_total_amount ,
    source.split_activity_amount,
    source.split_coupon_amount,
    source.operate_time 
);